# World Cup Match Predictor
## Notebook 2 / 5 — Feature Engineering

---

This is the most important notebook. The features we build here directly determine how accurate the model will be.

**We build 3 categories of features:**

| Category | What it measures | Why it matters |
|---|---|---|
| **Elo ratings** | Overall team strength, updated after every match | Best single predictor of match outcome |
| **Rolling form** | Goals scored/conceded, win rate in last 5 and 10 matches | Captures current momentum and fitness |
| **Head-to-head** | Historical record between these two specific teams | Some teams consistently beat others |

> **No data leakage:** Every feature only uses information available *before* the match. This is critical — leakage is the #1 reason beginners get fake high accuracy.

---
## Step 1 — Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

print('Libraries loaded!')

In [ ]:
# Load the clean data saved in Notebook 01
df = pd.read_csv('football_data_clean.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Add a unique match index — we need this later to merge features back correctly
df['match_idx'] = range(len(df))

print(f'Loaded {len(df):,} matches')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')

---
## Step 2 — Elo ratings

**Elo is a dynamic rating system.** Every team starts at 1500. After each match:
- The winner gains points, the loser loses points
- Beating a stronger team gives more points than beating a weak one
- Bigger wins (3-0) count more than narrow wins (1-0)
- World Cup matches update ratings more than friendlies (higher K-factor)

The Elo difference between two teams is the single strongest predictor of match outcome.

In [ ]:
# K-factor: controls how much ratings shift after each match
# Higher K = faster adaptation. World Cup matches matter most.
K_FACTORS = {
    'FIFA World Cup': 60,
    'Confederations Cup': 50,
    'UEFA Euro': 50,
    'Copa America': 50,
    'Africa Cup of Nations': 40,
    'AFC Asian Cup': 40,
    'CONCACAF Championship': 40,
    'UEFA Euro qualification': 40,
    'FIFA World Cup qualification': 40,
    'UEFA Nations League': 40,
    'Friendly': 20,
}

ELO_START = 1500  # All teams begin here

def expected_outcome(elo_a, elo_b):
    """Probability that team A wins, based on the Elo difference."""
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def goal_weight(goal_diff):
    """
    Bigger wins move the rating more, but with diminishing returns.
      1-goal win  -> weight 1.0
      2-goal win  -> weight 1.5
      3+ goal win -> weight (11 + diff) / 8
    """
    if goal_diff <= 1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    else:
        return (11 + goal_diff) / 8

k_wc  = K_FACTORS['FIFA World Cup']
k_wcq = K_FACTORS['FIFA World Cup qualification']
k_fri = K_FACTORS['Friendly']
print(f'K-factors -> World Cup: {k_wc}  |  Qualifier: {k_wcq}  |  Friendly: {k_fri}')
print('Elo functions ready!')

In [ ]:
# --- Main Elo loop ---
# We iterate through all matches in chronological order.
# For each match: record current ratings FIRST, then update them.
# This guarantees the stored rating is always pre-match (no leakage).

elo_ratings  = {}   # {team_name: current_elo}
home_elo_pre = []
away_elo_pre = []

print('Calculating Elo ratings for all matches...')

for _, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']

    # Get pre-match ratings (1500 for teams appearing for the first time)
    h_elo = elo_ratings.get(home, ELO_START)
    a_elo = elo_ratings.get(away, ELO_START)

    # Store BEFORE updating (no leakage)
    home_elo_pre.append(h_elo)
    away_elo_pre.append(a_elo)

    # Actual outcome
    if row['home_score'] > row['away_score']:
        act_h, act_a = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        act_h, act_a = 0.0, 1.0
    else:
        act_h, act_a = 0.5, 0.5

    # Expected outcome and update magnitude
    exp_h = expected_outcome(h_elo, a_elo)
    k     = K_FACTORS.get(row['tournament'], 30)
    gw    = goal_weight(abs(row['home_score'] - row['away_score']))

    # Update ratings
    elo_ratings[home] = h_elo + k * gw * (act_h - exp_h)
    elo_ratings[away] = a_elo + k * gw * (act_a - (1 - exp_h))

# Add to dataframe
df['home_elo']      = home_elo_pre
df['away_elo']      = away_elo_pre
df['elo_diff']      = df['home_elo'] - df['away_elo']
df['elo_prob_home'] = df.apply(
    lambda r: expected_outcome(r['home_elo'], r['away_elo']), axis=1
)

print(f'Done! Elo calculated for {len(elo_ratings)} teams.')

In [ ]:
# View current Elo rankings and chart the top 20
elo_df = (
    pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)

print('Top 20 teams by current Elo rating:\n')
print(elo_df.head(20).to_string(index=False))

In [ ]:
# Chart: Top 20 Elo rankings
top20 = elo_df.head(20)
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#534AB7' if i < 5 else '#8B84D7' if i < 10 else '#B5B0E8' for i in range(20)]
ax.barh(top20['team'][::-1], top20['elo'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(1500, color='red', linestyle='--', alpha=0.5, linewidth=1.2, label='Starting Elo (1500)')
ax.set_xlabel('Elo Rating', fontsize=12)
ax.set_title('Current International Elo Rankings — Top 20', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

for i, (_, row) in enumerate(top20[::-1].iterrows()):
    ax.text(row['elo'] + 3, i, f'{row["elo"]:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('elo_rankings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as elo_rankings.png')

---
## Step 3 — Rolling form statistics

Elo captures long-term team strength, but form captures **current momentum**. A team on a 5-game winning streak is very different from one on a 5-game losing streak, even if their Elo ratings are similar.

We compute rolling averages of:
- Goals scored and conceded
- Points per game (win=3, draw=1, loss=0)
- Win rate

For windows of the **last 5** and **last 10** matches.

> **Key technique:** We use `.shift(1)` before the rolling window. This shifts data by 1 match so the current match is never included in its own calculation.

In [ ]:
# Convert to 'long format': one row per team per match
# A team appears twice per match (once as home, once as away)

home_records = df[['match_idx', 'date', 'home_team', 'home_score', 'away_score', 'result']].copy()
home_records.columns = ['match_idx', 'date', 'team', 'goals_scored', 'goals_conceded', 'result']
home_records['won']     = (home_records['result'] == 'home_win').astype(int)
home_records['drew']    = (home_records['result'] == 'draw').astype(int)
home_records['lost']    = (home_records['result'] == 'away_win').astype(int)
home_records['is_home'] = True

away_records = df[['match_idx', 'date', 'away_team', 'away_score', 'home_score', 'result']].copy()
away_records.columns = ['match_idx', 'date', 'team', 'goals_scored', 'goals_conceded', 'result']
away_records['won']     = (away_records['result'] == 'away_win').astype(int)
away_records['drew']    = (away_records['result'] == 'draw').astype(int)
away_records['lost']    = (away_records['result'] == 'home_win').astype(int)
away_records['is_home'] = False

all_records = pd.concat([home_records, away_records], ignore_index=True)
all_records = all_records.sort_values(['team', 'date', 'match_idx']).reset_index(drop=True)
all_records['points'] = all_records['won'] * 3 + all_records['drew']

print(f'Long-format records: {len(all_records):,} rows')
print(f'Teams tracked: {all_records["team"].nunique()}')

In [ ]:
# Calculate rolling statistics
# .shift(1) ensures we only see matches BEFORE the current one (no leakage)
# min_periods=1 allows calculation even with only 1 previous match

print('Computing rolling form stats (this takes ~30 seconds)...')

for w in [5, 10]:
    all_records[f'gs_avg{w}'] = all_records.groupby('team')['goals_scored'].transform(
        lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean()
    )
    all_records[f'gc_avg{w}'] = all_records.groupby('team')['goals_conceded'].transform(
        lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean()
    )
    all_records[f'pts_avg{w}'] = all_records.groupby('team')['points'].transform(
        lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean()
    )
    all_records[f'win_rate{w}'] = all_records.groupby('team')['won'].transform(
        lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean()
    )

print('Done!')

# Sanity check: Brazil's recent records
print('\nBrazil last 5 records (form stats):')
br = all_records[all_records['team'] == 'Brazil'].tail(5)
print(br[['date', 'goals_scored', 'gs_avg5', 'won', 'pts_avg5', 'win_rate5']].to_string(index=False))

In [ ]:
# Merge rolling stats back to the main dataframe
# We use match_idx + is_home to uniquely identify each team-match record

feat_cols = [
    'gs_avg5', 'gc_avg5', 'pts_avg5', 'win_rate5',
    'gs_avg10', 'gc_avg10', 'pts_avg10', 'win_rate10',
]

home_feats = all_records[all_records['is_home'] == True][['match_idx'] + feat_cols].copy()
home_feats.columns = ['match_idx'] + [f'home_{c}' for c in feat_cols]

away_feats = all_records[all_records['is_home'] == False][['match_idx'] + feat_cols].copy()
away_feats.columns = ['match_idx'] + [f'away_{c}' for c in feat_cols]

df = df.merge(home_feats, on='match_idx', how='left')
df = df.merge(away_feats, on='match_idx', how='left')

# Fill any NaN (first 1-2 matches of a team's career) with the column mean
roll_cols = [c for c in df.columns if any(x in c for x in ['gs_avg', 'gc_avg', 'pts_avg', 'win_rate'])]
df[roll_cols] = df[roll_cols].fillna(df[roll_cols].mean())

print(f'Merged! DataFrame now has {len(df.columns)} columns.')
print(f'NaN in rolling cols: {df[roll_cols].isnull().sum().sum()}')

---
## Step 4 — Head-to-head (H2H) statistics

Some teams have a psychological advantage over specific opponents — regardless of overall strength. For example, Argentina historically dominates Brazil in World Cup knockout games.

For each match, we look at the **last 10 meetings** between those two specific teams and calculate:
- Win rate for the team playing as 'home'
- Average total goals in those meetings
- Number of previous meetings (confidence indicator)

> This cell takes **1-3 minutes** due to the pair lookup. The progress bar shows completion.

In [ ]:
print('Computing head-to-head stats (1-3 min)...')

# Create normalized team pair key (alphabetically sorted so A-B == B-A)
df['_pair'] = df.apply(
    lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1
)

h2h_data = {}  # {match_index: (win_rate, avg_goals, num_games)}

for pair in tqdm(df['_pair'].unique(), desc='Team pairs'):
    # All matches between this pair, sorted chronologically
    pair_df = df[df['_pair'] == pair].sort_values('date')
    running = []  # (home_team, result, total_goals) for past matches

    for orig_idx, row in pair_df.iterrows():
        if not running:
            # No prior meetings — use neutral defaults
            h2h_data[orig_idx] = (0.40, 2.5, 0)
        else:
            last_n = running[-10:]         # Last 10 meetings
            current_home = row['home_team']

            # Count wins for the current 'home' team in those past meetings
            wins = sum(
                1 for (mh, res, _) in last_n
                if (mh == current_home and res == 'home_win') or
                   (mh != current_home and res == 'away_win')
            )
            avg_g = float(np.mean([g for (_, _, g) in last_n]))

            h2h_data[orig_idx] = (wins / len(last_n), avg_g, len(last_n))

        # Add this match to running history (AFTER storing, so it's not used for itself)
        running.append((row['home_team'], row['result'], row['total_goals']))

# Map back to dataframe columns
df['h2h_home_win_rate'] = df.index.map(lambda i: h2h_data.get(i, (0.40, 2.5, 0))[0])
df['h2h_avg_goals']     = df.index.map(lambda i: h2h_data.get(i, (0.40, 2.5, 0))[1])
df['h2h_num_games']     = df.index.map(lambda i: h2h_data.get(i, (0.40, 2.5, 0))[2])

df = df.drop(columns=['_pair'])
print('H2H stats computed!')

In [ ]:
# Sanity check: Brazil vs Argentina H2H history
bra_arg = df[
    ((df['home_team'] == 'Brazil')    & (df['away_team'] == 'Argentina')) |
    ((df['home_team'] == 'Argentina') & (df['away_team'] == 'Brazil'))
][['date', 'home_team', 'home_score', 'away_score', 'away_team',
   'h2h_home_win_rate', 'h2h_avg_goals', 'h2h_num_games']].tail(8)

print('Brazil vs Argentina — last 8 meetings with H2H features:')
print(bra_arg.to_string(index=False))

---
## Step 5 — Final features and targets

We add two final context features and then define the complete feature and target column lists that all future notebooks will use.

In [ ]:
# Tournament importance weight (0 to 1)
TOURNAMENT_WEIGHT = {
    'FIFA World Cup': 1.00,
    'Confederations Cup': 0.85,
    'UEFA Euro': 0.85,
    'Copa America': 0.85,
    'Africa Cup of Nations': 0.70,
    'AFC Asian Cup': 0.70,
    'CONCACAF Championship': 0.70,
    'UEFA Euro qualification': 0.60,
    'FIFA World Cup qualification': 0.60,
    'UEFA Nations League': 0.55,
    'Friendly': 0.30,
}
df['tournament_weight'] = df['tournament'].map(TOURNAMENT_WEIGHT).fillna(0.50)

# Encode result as a number for the classifier
# 0 = away win, 1 = draw, 2 = home win
RESULT_LABEL = {'away_win': 0, 'draw': 1, 'home_win': 2}
df['result_label'] = df['result'].map(RESULT_LABEL)

# --- Define feature and target columns ---

FEATURE_COLS = [
    # Elo-based (4 features)
    'home_elo', 'away_elo', 'elo_diff', 'elo_prob_home',
    # Home team rolling form — last 5 and 10 matches (8 features)
    'home_gs_avg5',  'home_gc_avg5',  'home_pts_avg5',  'home_win_rate5',
    'home_gs_avg10', 'home_gc_avg10', 'home_pts_avg10', 'home_win_rate10',
    # Away team rolling form — last 5 and 10 matches (8 features)
    'away_gs_avg5',  'away_gc_avg5',  'away_pts_avg5',  'away_win_rate5',
    'away_gs_avg10', 'away_gc_avg10', 'away_pts_avg10', 'away_win_rate10',
    # Head-to-head (3 features)
    'h2h_home_win_rate', 'h2h_avg_goals', 'h2h_num_games',
    # Match context (2 features)
    'is_neutral', 'tournament_weight',
]

TARGET_COLS = [
    'result',       # Match winner (string: home_win / draw / away_win)
    'result_label', # Match winner (int: 2 / 1 / 0) — for XGBoost classifier
    'home_score',   # Exact goals home team scored — for Poisson model
    'away_score',   # Exact goals away team scored — for Poisson model
    'total_goals',  # Total goals — for Over/Under market
    'over_2_5',     # 1 if total goals > 2.5 — binary classification
    'btts',         # 1 if both teams scored — binary classification
]

print(f'Total features: {len(FEATURE_COLS)}')
for i, f in enumerate(FEATURE_COLS):
    print(f'  {i+1:2d}. {f}')
print(f'\nTargets: {TARGET_COLS}')

---
## Step 6 — Validate features and check for leakage

In [ ]:
# Drop the small number of rows with NaN in any feature
df_model = df.dropna(subset=FEATURE_COLS + ['result_label']).copy()

print(f'Rows before dropping NaN: {len(df):,}')
print(f'Rows after dropping NaN:  {len(df_model):,}')
print(f'Rows dropped:             {len(df) - len(df_model):,}')
print()
print('Feature statistics:')
print(df_model[FEATURE_COLS].describe().round(2))

In [ ]:
# Correlation of each feature with the match result
# Positive = helps the home team win, Negative = helps the away team win

correlations = df_model[FEATURE_COLS].corrwith(df_model['result_label'])
correlations_sorted = correlations.abs().sort_values(ascending=False)

print('Features ranked by correlation with match result:\n')
for feat, val in correlations_sorted.items():
    direction = '+' if correlations[feat] >= 0 else '-'
    bar = '#' * int(abs(val) * 40)
    print(f'  {direction} {feat:<25s}  {val:.3f}  {bar}')

In [ ]:
# Chart: Feature correlation with result
corr_vals = df_model[FEATURE_COLS].corrwith(df_model['result_label']).sort_values()
colors = ['#D85A30' if v < 0 else '#1D9E75' for v in corr_vals]

fig, ax = plt.subplots(figsize=(10, 9))
corr_vals.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(
    'Feature correlation with match result\n'
    'Green = helps home team win  |  Red = helps away team win',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Pearson correlation coefficient')
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as feature_correlations.png')

---
## Step 7 — Save

In [ ]:
# Save full feature dataset
df_model.to_csv('features_dataset.csv', index=False)

# Save current Elo ratings (needed in production to rate incoming teams)
elo_df.to_csv('current_elo_ratings.csv', index=False)

# Save column config as JSON so all future notebooks use exactly the same columns
config = {
    'feature_cols': FEATURE_COLS,
    'target_cols': TARGET_COLS,
    'result_label_map': RESULT_LABEL,
    'k_factors': K_FACTORS,
    'tournament_weights': TOURNAMENT_WEIGHT,
    'elo_start': ELO_START,
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Files saved:')
print(f'  features_dataset.csv    {len(df_model):>6,} rows  x  {len(FEATURE_COLS)} features')
print(f'  current_elo_ratings.csv {len(elo_df):>6,} teams')
print(f'  model_config.json       column and config reference for all future notebooks')
print()
print('Up next: Notebook 03 — Model Training')
print('We will train XGBoost for match winner and Poisson for goal counts.')

---
## Summary

| Feature group | # features | Key insight |
|---|---|---|
| Elo ratings | 4 | `elo_diff` is typically the strongest single predictor |
| Rolling form (home) | 8 | Last 5 matches often more predictive than last 10 |
| Rolling form (away) | 8 | Away team's scoring form predicts Over/Under well |
| Head-to-head | 3 | Matters most for long-standing rivalries |
| Match context | 2 | `is_neutral` heavily impacts home team advantage |
| **Total** | **25** | |

**What to expect from the correlation chart:**
- `elo_diff` and `elo_prob_home` should show the highest correlation (teams with better Elo win more)
- `home_pts_avg5` and `home_win_rate5` should be clearly positive
- `away_pts_avg5` and `away_win_rate5` should be negative (a strong away team hurts the home win probability)
- `is_neutral` should be negative (neutral ground removes home advantage)

---